# Exploratory Data Analysis (EDA)

Notebook này phân tích dữ liệu ban đầu:
- Load và giải nén data
- Đếm số lượng ảnh mỗi class
- Phát hiện class imbalance
- Hiển thị samples

## Setup & Load Config

In [ ]:
# Import config
import sys
sys.path.append('..')  # Thêm parent directory vào path

from config import config

# Import libraries
import os
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.image as mpimg
import random

# Print config
config.print_config()

## Load Data

In [ ]:
# Mount Drive (chỉ trên Colab)
if config.IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Unzip data
    if os.path.exists(config.DRIVE_PATH):
        with zipfile.ZipFile(config.DRIVE_PATH, 'r') as zip_ref:
            zip_ref.extractall(config.DATA_DIR)
        print(" Giải nén thành công!")
    else:
        print(f" Không tìm thấy {config.DRIVE_PATH}")
else:
    print("  Chạy trên local - giả định data đã có sẵn")

# List categories
categories = [d for d in os.listdir(config.DATA_DIR) if os.path.isdir(os.path.join(config.DATA_DIR, d))]
print(f"\n Tìm thấy {len(categories)} categories: {categories}")

## Phân tích số lượng ảnh

In [ ]:
# Đếm số lượng ảnh trong mỗi lớp
data_stats = {}

for cat in categories:
    path = os.path.join(config.DATA_DIR, cat)
    if os.path.isdir(path):
        data_stats[cat] = len(os.listdir(path))

# Vẽ biểu đồ phân bổ
plt.figure(figsize=(12, 5))
sns.barplot(x=list(data_stats.keys()), y=list(data_stats.values()))
plt.title('Số lượng ảnh trong mỗi loại rác thải')
plt.xlabel('Category')
plt.ylabel('Number of Images')
plt.xticks(rotation=45)
plt.show()

# In tổng số ảnh
total_images = sum(data_stats.values())
print(f"\n Tổng số ảnh: {total_images:,}")

## Phát hiện Class Imbalance

In [ ]:
# Tạo DataFrame từ data_stats để dễ quan sát tỉ lệ
stats_df = pd.DataFrame(list(data_stats.items()), columns=['Category', 'Count'])
stats_df['Percentage'] = (stats_df['Count'] / stats_df['Count'].sum() * 100).round(2)
stats_df = stats_df.sort_values(by='Count', ascending=False)

print("Thống kê chi tiết độ mất cân bằng:")
display(stats_df)

# Tính toán hệ số mất cân bằng (giữa lớp nhiều nhất và ít nhất)
imbalance_ratio = stats_df['Count'].max() / stats_df['Count'].min()
print(f"\n  Lớp nhiều nhất gấp {imbalance_ratio:.2f} lần lớp ít nhất.")

if imbalance_ratio > 3:
    print("\n Cảnh báo: Dữ liệu đang bị mất cân bằng nghiêm trọng!")
    print(" Khuyến nghị: Cần cân bằng dữ liệu bằng augmentation hoặc undersampling.")
    print(f"   Xem notebook 02_data_balancing.ipynb để xử lý.")
else:
    print("\n Dữ liệu tương đối cân bằng.")

## Hiển thị Sample Images

In [ ]:
# Hiển thị 5 ảnh ngẫu nhiên từ các lớp
plt.figure(figsize=(20, 10))
for i in range(5):
    random_cat = random.choice(categories)
    cat_path = os.path.join(config.DATA_DIR, random_cat)
    random_img = random.choice(os.listdir(cat_path))
    img_path = os.path.join(cat_path, random_img)

    img = mpimg.imread(img_path)
    plt.subplot(1, 5, i + 1)
    plt.imshow(img)
    plt.title(random_cat)
    plt.axis('off')

plt.tight_layout()
plt.show()

## Kết luận

- Đã load và phân tích dữ liệu
- Đã phát hiện class imbalance
- **Next step**: Chạy `02_data_balancing.ipynb` để cân bằng dữ liệu